<a href="https://colab.research.google.com/github/Mwimwii/lvm/blob/main/lecture_audio_plain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture Audio + Video Generator

Generates a full lecture video from:
- A `script.json` file (slide text)
- A PDF of slides (one page per slide)

**Script format:**
```json
{
  "1": ["Slide 1 sentence one.", "Slide 1 sentence two."],
  "2": ["Slide 2 sentence one."]
}
```

**Pipeline:**
```
script.json  →  [VoxCPM2]  →  slides_audio/slide_001.wav
slides.pdf   →  [PyMuPDF]  →  slides_images/slide_001.png
                             ↓
               [FFmpeg: image + audio → clip]
                             ↓
               [FFmpeg: concat all clips → lecture.mp4]
```

In [1]:
# ── Cell 1: Install all dependencies ─────────────────────────────────────────
!pip install voxcpm soundfile numpy pymupdf --quiet
# ffmpeg is pre-installed on Colab; verify:
!ffmpeg -version 2>&1 | head -1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 10.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 32.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 145.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ── Cell 2: Clone reference voice repo ───────────────────────────────────────
import os

if not os.path.exists('vo'):
    !git clone https://github.com/Mwimwii/vo
else:
    print('vo/ already present, skipping clone')

REF_PROMPT_WAV = '/content/vo/speaker.wav'
REF_TEXT = (
    "A string is defined as a sequence of characters, and it is important to "
    "remember that these are case sensitive. Strings can contain anything from "
    "standard alphabet letters, the special symbols, spaces and numerical digits."
)
print('Reference audio ready:', REF_PROMPT_WAV)

Cloning into 'vo'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 1.39 MiB | 4.37 MiB/s, done.
Reference audio ready: /content/vo/speaker.wav


In [3]:
import torch
import numpy as np
import soundfile as sf
import os
from voxcpm import VoxCPM
import torch

print('Loading VoxCPM2 on T4 GPU...')
model = VoxCPM.from_pretrained(
    'openbmb/VoxCPM2',
)

Loading VoxCPM2 on T4 GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

voxcpm_model_path: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d, zipenhancer_model_path: iic/speech_zipenhancer_ans_multiloss_16k_base, enable_denoiser: True
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d/audiovae.pth
Running on device: cuda, dtype: bfloat16
Loading model from safetensors: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d/model.safetensors
Loaded VoxCPM2Model


2026-04-21 12:18:45,013 - modelscope - INFO - Got 9 files, start to download ...


Processing 9 items:   0%|          | 0.00/9.00 [00:00<?, ?it/s]

2026-04-21 12:18:48,050 - modelscope - INFO - Finish downloading 9 files for repo 'iic/speech_zipenhancer_ans_multiloss_16k_base'
2026-04-21 12:18:48,055 - modelscope - INFO - initiate model from /root/.cache/modelscope/hub/models/iic/speech_zipenhancer_ans_multiloss_16k_base
2026-04-21 12:18:48,056 - modelscope - INFO - initiate model from location /root/.cache/modelscope/hub/models/iic/speech_zipenhancer_ans_multiloss_16k_base.
2026-04-21 12:18:48,058 - modelscope - INFO - initialize model from /root/.cache/modelscope/hub/models/iic/speech_zipenhancer_ans_multiloss_16k_base
2026-04-21 12:18:48,294 - modelscope - WARNING - No preprocessor field found in cfg.
2026-04-21 12:18:48,295 - modelscope - WARNING - No val key and type key found in preprocessor domain of configuration.json file.
2026-04-21 12:18:48,295 - modelscope - WARNING - Cannot find available config to build preprocessor at mode inference, current config: {'model_dir': '/root/.cache/modelscope/hub/models/iic/speech_zipenh

In [4]:
# ── Cell 4: Configuration ─────────────────────────────────────────────────────
# ── Global Behavior ──
CLEAR_ASSETS = True    # Set to True to delete existing audio/video and start fresh

# ── Audio ──
CFG             = 2.7    # Voice guidance (higher = closer to reference voice)
TIMESTEPS       = 10     # Inference steps (higher = better, slower)
SAMPLE_RATE     = 48000  # Hz

GAP_BETWEEN_SEGMENTS = 0.3   # seconds of silence between sentences in a slide
GAP_BETWEEN_SLIDES   = 1.0   # seconds of silence between slides

# ── Video ──
VIDEO_HEIGHT    = 720    # Output height in pixels (width auto-scaled)
VIDEO_FPS       = 10     # Frames per second (static slides → low FPS is fine)
AUDIO_BITRATE   = '192k' # AAC audio bitrate

# ── Paths ──
SLIDES_AUDIO_DIR  = 'slides_audio'
SLIDES_IMAGES_DIR = 'slides_images'
SLIDES_VIDEO_DIR  = 'slides_video'
AUDIO_OUTPUT      = 'lecture.wav'
VIDEO_OUTPUT      = 'lecture.mp4'

for d in [SLIDES_AUDIO_DIR, SLIDES_IMAGES_DIR, SLIDES_VIDEO_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Config ready. CLEAR_ASSETS is {CLEAR_ASSETS}')

Config ready. CLEAR_ASSETS is True


In [5]:
# ── Cell 5: Upload script.json ────────────────────────────────────────────────
import json
from google.colab import files

print('Upload your script.json file:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]
script = json.loads(uploaded[filename])

# Sort numerically so slide 10 comes after 9, not after 1
slide_keys = sorted(script.keys(), key=lambda k: int(k))

print(f'\nLoaded: {filename}')
print(f'Slides: {len(slide_keys)}  ({slide_keys[0]} → {slide_keys[-1]})')
print(f'Total segments: {sum(len(script[k]) for k in slide_keys)}')

Upload your script.json file:


Saving slide.json to slide.json

Loaded: slide.json
Slides: 15  (1 → 15)
Total segments: 70


In [6]:
# ── Cell 6: Upload slides PDF and convert to images ───────────────────────────
import fitz  # PyMuPDF

# Clear images directory if requested
if CLEAR_ASSETS:
    print(f'CLEAR_ASSETS is True: cleaning {SLIDES_IMAGES_DIR}...')
    for f in os.listdir(SLIDES_IMAGES_DIR):
        if f.endswith('.png'):
            os.remove(os.path.join(SLIDES_IMAGES_DIR, f))

print('Upload your slides PDF:')
uploaded_pdf = files.upload()

pdf_filename = list(uploaded_pdf.keys())[0]

# Save to disk (Colab files.upload() returns bytes)
pdf_path = os.path.join('/content', pdf_filename)
with open(pdf_path, 'wb') as f:
    f.write(uploaded_pdf[pdf_filename])

# Convert every page to a PNG named slide_001.png, slide_002.png …
doc = fitz.open(pdf_path)
print(f'\nPDF: {pdf_filename}  ({len(doc)} pages)')

for page_num in range(len(doc)):
    slide_num = page_num + 1           # 1-based to match script keys
    page = doc[page_num]
    # mat: scale factor — 2.0 gives ~1920px wide for a standard 16:9 slide
    mat = fitz.Matrix(2.0, 2.0)
    pix = page.get_pixmap(matrix=mat)
    out_path = os.path.join(SLIDES_IMAGES_DIR, f'slide_{str(slide_num).zfill(3)}.png')
    pix.save(out_path)

doc.close()

image_files = sorted([f for f in os.listdir(SLIDES_IMAGES_DIR) if f.endswith('.png')])
print(f'Saved {len(image_files)} PNG images to {SLIDES_IMAGES_DIR}/')

# ── Matching report ──────────────────────────────────────────────────────────
print('\nSlide matching check:')
pdf_nums  = set(range(1, len(image_files) + 1))
script_nums = set(int(k) for k in slide_keys)
matched   = pdf_nums & script_nums
img_only  = pdf_nums - script_nums
audio_only = script_nums - pdf_nums
print(f'  Matched: {len(matched)} slide(s)')
if img_only:
    print(f'  PDF only (no audio): {sorted(img_only)}')
if audio_only:
    print(f'  Script only (no image): {sorted(audio_only)}')

CLEAR_ASSETS is True: cleaning slides_images...
Upload your slides PDF:


Saving slide.pdf to slide.pdf

PDF: slide.pdf  (15 pages)
Saved 15 PNG images to slides_images/

Slide matching check:
  Matched: 15 slide(s)


In [7]:
# ── Cell 7: Generate per-slide audio ─────────────────────────────────────────
# Clear audio directory if requested
if CLEAR_ASSETS:
    print(f'CLEAR_ASSETS is True: cleaning {SLIDES_AUDIO_DIR}...')
    for f in os.listdir(SLIDES_AUDIO_DIR):
        if f.endswith('.wav'):
            os.remove(os.path.join(SLIDES_AUDIO_DIR, f))

def make_silence(seconds):
    return np.zeros(int(seconds * SAMPLE_RATE))

def generate_segment(text):
    return model.generate(
        text=text,
        prompt_wav_path=REF_PROMPT_WAV,
        prompt_text=REF_TEXT,
        cfg_value=CFG,
        inference_timesteps=TIMESTEPS,
        normalize=True,
        denoise=True,
        retry_badcase=True,
        retry_badcase_max_times=3,
        retry_badcase_ratio_threshold=6.0,
    )

def process_slide_audio(slide_num):
    wav_path = os.path.join(SLIDES_AUDIO_DIR, f'slide_{slide_num.zfill(3)}.wav')

    # Check for existing file if not clearing assets
    if not CLEAR_ASSETS and os.path.exists(wav_path):
        print(f'Slide {slide_num}: already exists, loading.')
        slide_wav, _ = sf.read(wav_path)
        return slide_num, slide_wav

    segments = script[slide_num]
    print(f'\n[Start] Generating Slide {slide_num} ({len(segments)} segment(s))')
    chunks = []
    seg_silence = make_silence(GAP_BETWEEN_SEGMENTS)

    for idx, text in enumerate(segments):
        wav = generate_segment(text)
        chunks.append(wav)
        if idx < len(segments) - 1:
            chunks.append(seg_silence)

    slide_wav = np.concatenate(chunks)
    sf.write(wav_path, slide_wav, SAMPLE_RATE)
    print(f'[Done] Slide {slide_num} audio saved.')
    return slide_num, slide_wav

print(f'Generating audio for {len(slide_keys)} slides sequentially...')

results = []
for slide_num in slide_keys:
    results.append(process_slide_audio(slide_num))

# Sort results to maintain correct lecture order
results.sort(key=lambda x: int(x[0]))

all_audio = []
slide_silence = make_silence(GAP_BETWEEN_SLIDES)
for slide_num, slide_wav in results:
    all_audio.extend([slide_wav, slide_silence])

full_audio = np.concatenate(all_audio)
sf.write(AUDIO_OUTPUT, full_audio, SAMPLE_RATE)
print(f'\nFull audio saved: {AUDIO_OUTPUT}')

CLEAR_ASSETS is True: cleaning slides_audio...
Generating audio for 15 slides sequentially...

[Start] Generating Slide 1 (3 segment(s))
inputs:(1, 243111)
decode_do_segement
padding: 4889
inputs after padding:(1, 248000)
current_idx: 240000 100.00%


 29%|██▉       | 15/52 [00:05<00:13,  2.68it/s]


inputs:(1, 243111)
decode_do_segement
padding: 4889
inputs after padding:(1, 248000)
current_idx: 240000 100.00%


 25%|██▌       | 39/154 [00:14<00:41,  2.74it/s]


inputs:(1, 243111)
decode_do_segement
padding: 4889
inputs after padding:(1, 248000)
current_idx: 240000 100.00%


...

inputs:(1, 243111)
decode_do_segement
padding: 4889
inputs after padding:(1, 248000)
current_idx: 240000 100.00%


 26%|██▋       | 20/76 [00:07<00:21,  2.61it/s]


inputs:(1, 243111)
decode_do_segement
padding: 4889
inputs after padding:(1, 248000)
current_idx: 240000 100.00%


 35%|███▌      | 46/130 [00:16<00:30,  2.72it/s]


[Done] Slide 15 audio saved.

Full audio saved: lecture.wav


In [8]:
# ── Cell 8: Generate per-slide video clips (image + audio → MP4) ──────────────
import subprocess
import os
from concurrent.futures import ThreadPoolExecutor

# Clear video directory if requested
if CLEAR_ASSETS:
    print(f'CLEAR_ASSETS is True: cleaning {SLIDES_VIDEO_DIR}...')
    for f in os.listdir(SLIDES_VIDEO_DIR):
        if f.endswith('.mp4'):
            os.remove(os.path.join(SLIDES_VIDEO_DIR, f))

def make_slide_clip_worker(args):
    num, image_path, audio_path, output_path = args

    # If not clearing and clip exists, skip FFmpeg
    if not CLEAR_ASSETS and os.path.exists(output_path):
        return (num, output_path)

    if os.path.exists(output_path):
        os.remove(output_path)

    print(f'  [Start] Rendering Slide {num}...')
    cmd = [
        'ffmpeg', '-y', '-loop', '1', '-i', image_path, '-i', audio_path,
        '-c:v', 'libx264', '-tune', 'stillimage', '-vf', f'scale=-2:{VIDEO_HEIGHT},format=yuv420p',
        '-c:a', 'aac', '-b:a', AUDIO_BITRATE, '-r', str(VIDEO_FPS), '-shortest', output_path
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print(f'  [Done] Slide {num} rendered.')
        return (num, output_path)
    else:
        print(f'  [Error] Slide {num} failed.')
        return None

available_images = {int(f.split("_")[1].split(".")[0]) for f in os.listdir(SLIDES_IMAGES_DIR) if f.endswith(".png")}
available_audio = {int(f.split("_")[1].split(".")[0]) for f in os.listdir(SLIDES_AUDIO_DIR) if f.endswith(".wav")}
matched_slides = sorted(available_images & available_audio)

tasks = []
for num in matched_slides:
    key = str(num).zfill(3)
    tasks.append((num, os.path.join(SLIDES_IMAGES_DIR, f'slide_{key}.png'), os.path.join(SLIDES_AUDIO_DIR, f'slide_{key}.wav'), os.path.join(SLIDES_VIDEO_DIR, f'slide_{key}.mp4')))

print(f'Generating video clips for {len(matched_slides)} slide(s) concurrently using 4 workers...')
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(make_slide_clip_worker, tasks))

successful_tasks = sorted([r for r in results if r is not None], key=lambda x: x[0])
clip_paths = [path for num, path in successful_tasks]
print(f'\n{len(clip_paths)} clips ready and sorted.')

CLEAR_ASSETS is True: cleaning slides_video...
Generating video clips for 15 slide(s) concurrently using 4 workers...
  [Start] Rendering Slide 1...
  [Start] Rendering Slide 2...
  [Start] Rendering Slide 3...
  [Start] Rendering Slide 4...
  [Done] Slide 1 rendered.
  [Start] Rendering Slide 5...
  [Done] Slide 4 rendered.
  [Start] Rendering Slide 6...
  [Done] Slide 2 rendered.
  [Start] Rendering Slide 7...
  [Done] Slide 3 rendered.
  [Start] Rendering Slide 8...
  [Done] Slide 6 rendered.
  [Start] Rendering Slide 9...
  [Done] Slide 8 rendered.
  [Start] Rendering Slide 10...
  [Done] Slide 7 rendered.
  [Start] Rendering Slide 11...
  [Done] Slide 5 rendered.
  [Start] Rendering Slide 12...
  [Done] Slide 9 rendered.
  [Start] Rendering Slide 13...
  [Done] Slide 10 rendered.
  [Start] Rendering Slide 14...
  [Done] Slide 11 rendered.
  [Start] Rendering Slide 15...
  [Done] Slide 12 rendered.
  [Done] Slide 13 rendered.
  [Done] Slide 14 rendered.
  [Done] Slide 15 rendered.


In [9]:
# ── Cell 9: Concatenate all clips into final lecture.mp4 ──────────────────────

concat_list = '/content/concat_list.txt'
with open(concat_list, 'w') as f:
    for clip in clip_paths:
        f.write(f"file '{clip}'\n")

cmd = [
    'ffmpeg', '-y',
    '-f', 'concat',
    '-safe', '0',
    '-i', concat_list,
    '-c', 'copy',
    VIDEO_OUTPUT
]

print(f'Concatenating {len(clip_paths)} clips → {VIDEO_OUTPUT} ...')
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    size_mb = os.path.getsize(VIDEO_OUTPUT) / 1024 / 1024
    print(f'Done. {VIDEO_OUTPUT} ({size_mb:.1f} MB)')
else:
    print('FFmpeg error:')
    print(result.stderr[-500:])

os.remove(concat_list)

Concatenating 15 clips → lecture.mp4 ...
Done. lecture.mp4 (8.2 MB)


In [10]:
# ── Cell 10: Preview and download ────────────────────────────────────────────
from IPython.display import Video, Audio, display

print('Preview (first 30 seconds):')
display(Video(VIDEO_OUTPUT, width=720))

print('\nDownloading...')
files.download(VIDEO_OUTPUT)

Preview (first 30 seconds):



Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── Cell 11: (Optional) Preview / re-render a single slide ───────────────────
from IPython.display import Video, display

PREVIEW_SLIDE = '1'   # ← change to any slide number

key   = PREVIEW_SLIDE.zfill(3)
clip  = os.path.join(SLIDES_VIDEO_DIR, f'slide_{key}.mp4')
img   = os.path.join(SLIDES_IMAGES_DIR, f'slide_{key}.png')
audio = os.path.join(SLIDES_AUDIO_DIR,  f'slide_{key}.wav')

if not os.path.exists(clip):
    print(f'Clip not found — regenerating slide {PREVIEW_SLIDE}...')
    make_slide_clip(img, audio, clip)

print(f'Slide {PREVIEW_SLIDE}:')
display(Video(clip, width=720))